In [1]:
"""
exp06 Step 01: Capped-Regret Re-Entry Exploration (EXPLORATION ON 2005-2014 ONLY; context, PROTOCOL.md §9)

The 12 candidates of PROTOCOL.md §5 (exit on the trend signal or near the all-time high; buy back at once if the close
rises b above the sale, with a cooling-off of c decisions, or on a recovery above the 200-session average after a close
below it) are replayed as ONE continuous path over 2005-01-03 -> 2014-12-31 (start invested, no forced buy-back), against
buy-and-hold over the same sessions. The data is cut at so.config.EXPLORATION_DATA_CUTOFF_DATE_STR (2015-03-18).

What it shows: how often each exit is wrong (ended by the buy-stop) and the roadmap arithmetic (1 - q) x G vs q x |L|.
DATA-DEPENDENT FOLLOW-UP of exp03; 2008 is known to everyone (hindsight-prone). The result does not change the grid or
the selection (pre-registered). Outputs: store04_experiments/exp06_capped_regret_reentry/step01_exploration_data/.
Trials: 12 (stage "exploration").
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp06_capped_regret_reentry import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE CONTINUOUS REPLAY FUNCTIONS
from so.core.continuous_replay import simulate_continuous_replay_dict, get_episode_scorecard_pdf, get_replay_summary_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp06_capped_regret_reentry.rules import get_rule_candidate_list, get_rule_builder_func, get_trigger_summary_dict
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_exploration_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "exploration")

Experiment:	exp06_capped_regret_reentry	Protocol:	1.0	Config Hash:	5927a03c1d
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp06_capped_regret_reentry/step01_exploration_data/
ℹ️ No earlier 'exploration' entry of exp06_capped_regret_reentry in the trial log (config hash 5927a03c1d).


In [4]:
# CALL FUNCTION TO READ THE MINUTE BARS UP TO THE EXPLORATION DATA CUTOFF (NOTHING FROM THE WALK-FORWARD PERIOD IS READ)
complete_ohlcv_pdf = get_complete_ohlcv_pdf(cutoff_date_str_in=config.EXPLORATION_DATA_CUTOFF_DATE_STR)
# CALL FUNCTION TO CONVERT THE DATA TO ARRAYS AND BUILD THE DAILY TABLE
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# DEFINE THE EXPLORATION WINDOW
exploration_start_str, exploration_end_str = config.EXPLORATION_START_DATE_STR, config.EXPLORATION_END_DATE_STR
# DEFINE ONE PERIOD PER CALENDAR YEAR (FOR THE YEARS-WON COUNT)
session_date_arr = np.array(ohlcv_array_dict["session_date_list"])
window_date_arr = session_date_arr[(session_date_arr >= pd.Timestamp(exploration_start_str).date()) & (session_date_arr <= pd.Timestamp(exploration_end_str).date())]
year_period_pdf = pd.DataFrame([{"period_id": year, "period_start": min(d for d in window_date_arr if d.year == year), "period_end": max(d for d in window_date_arr if d.year == year)}
                                for year in sorted({d.year for d in window_date_arr})])
# DISPLAY INFORMATION
print(f"Data:\t{daily_pdf['date'].min()} -> {daily_pdf['date'].max()}\t({len(daily_pdf):,} sessions)\tReplay:\t{exploration_start_str} -> {exploration_end_str}\t({len(window_date_arr):,} sessions, {len(year_period_pdf)} years)")

Data:	2005-01-03 -> 2015-03-18	(2,568 sessions)	Replay:	2005-01-03 -> 2014-12-31	(2,516 sessions, 10 years)


In [5]:
"""
The 12 candidates, each one continuous path (start invested, no forced buy-back), against buy-and-hold.
"""
# SIMULATE BUY AND HOLD
buy_hold_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str)
print(f"Buy & hold:\ttotal return {buy_hold_dict['metric_dict']['total_return']:.2%} | Sharpe {buy_hold_dict['metric_dict']['sharpe_ratio']:.2f} | max drawdown {buy_hold_dict['metric_dict']['max_drawdown']:.2%}")
# DEFINE THE RULE BUILDER
rule_builder_func = get_rule_builder_func(daily_pdf)
# LISTS TO HOLD THE RESULTS
row_dict_list, scorecard_pdf_list = [], []
# ITERATE OVER THE CANDIDATES
for candidate_idx, candidate_dict in enumerate(get_rule_candidate_list()):
    # BUILD AND REPLAY THE CANDIDATE
    exit_signal_arr, reentry_func, reason_str = rule_builder_func(candidate_dict)
    simulation_dict = simulate_continuous_replay_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str, exp_config.UNTOUCHED_START_DATE_STR,
                                                      volatility_arr_in=np.full(len(daily_pdf), np.nan), stop_k_in=None, reentry_func_in=reentry_func,
                                                      reentry_reason_str_in=reason_str, exit_signal_arr_in=exit_signal_arr, max_cash_sessions_in=exp_config.MAX_CASH_SESSIONS)
    # SUMMARIZE THE PATH (YEARS AS PERIODS) AND THE TRIGGERS
    summary_dict = get_replay_summary_dict(simulation_dict["daily_equity_pdf"], buy_hold_dict["daily_equity_pdf"], year_period_pdf, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
    trigger_dict = get_trigger_summary_dict(simulation_dict)
    scorecard_pdf = get_episode_scorecard_pdf(simulation_dict, ohlcv_array_dict)
    scorecard_pdf_list.append(scorecard_pdf.assign(candidate_idx=candidate_idx, **candidate_dict))
    # STORE THE ROW
    row_dict_list.append({"candidate_idx": candidate_idx, **candidate_dict, "total_return": summary_dict["strategy_total_return"], "buy_hold_total_return": summary_dict["buy_hold_total_return"],
                          "excess_total_return": summary_dict["excess_total_return"], "annualized_return": summary_dict["strategy_annualized_return"],
                          "sharpe_ratio": summary_dict["strategy_sharpe_ratio"], "max_drawdown": summary_dict["strategy_max_drawdown"], "in_market_pct": summary_dict["in_market_pct"],
                          "exit_count": simulation_dict["metric_dict"]["signal_exit_count"], "mean_cash_sessions": simulation_dict["metric_dict"]["mean_cash_sessions"],
                          "bought_back_lower_pct": simulation_dict["metric_dict"]["positive_episode_pct"], **trigger_dict,
                          "years_won": summary_dict["period_windows_beating_buy_hold"], "year_count": summary_dict["period_window_count"],
                          "annualized_log_excess": summary_dict["log_excess_annualized_log_excess"], "log_excess_ci_low": summary_dict["log_excess_ci_low"], "log_excess_ci_high": summary_dict["log_excess_ci_high"]})
# CONVERT THE RESULTS
exploration_pdf = pd.DataFrame(row_dict_list)
exploration_scorecard_pdf = pd.concat(scorecard_pdf_list, ignore_index=True)
# DISPLAY THE EXCESS RETURN GRID (EXIT SIGNAL AND BUY-STOP x COOLING-OFF)
display(exploration_pdf.pivot_table(index=["exit_signal", "buy_stop"], columns="cooling", values="excess_total_return").round(4))
# PREVIEW DATAFRAME
exploration_pdf[["candidate_idx", "exit_signal", "buy_stop", "cooling", "total_return", "buy_hold_total_return", "excess_total_return", "in_market_pct", "exit_count",
                 "buy_stop_count", "recovery_count", "wrong_exit_share", "mean_gain_right", "mean_loss_wrong", "right_side", "wrong_side", "max_drawdown",
                 "annualized_log_excess", "log_excess_ci_low", "log_excess_ci_high"]].round(4)

Buy & hold:	total return 69.02% | Sharpe 0.36 | max drawdown -56.44%


cooling                   0       20
exit_signal buy_stop                
ath_1pct    0.01     -0.4819 -0.3923
            0.02     -0.4271 -0.3358
            0.03     -0.4443 -0.3043
trend_ma200 0.01     -0.2459 -0.7846
            0.02     -0.1951  0.0959
            0.03     -0.1862 -0.1825

,candidate_idx,exit_signal,buy_stop,cooling,total_return,buy_hold_total_return,excess_total_return,in_market_pct,exit_count,buy_stop_count,recovery_count,wrong_exit_share,mean_gain_right,mean_loss_wrong,right_side,wrong_side,max_drawdown,annualized_log_excess,log_excess_ci_low,log_excess_ci_high
0,0,trend_ma200,0.01,0,0.4443,0.6902,-0.2459,0.7594,42,31,11,0.7381,0.0344,-0.0172,0.0090,0.0127,-0.2320,-0.0157,-0.1101,0.1018
1,1,trend_ma200,0.01,20,-0.0944,0.6902,-0.7846,0.9038,34,27,7,0.7941,-0.0052,-0.0217,-0.0011,0.0172,-0.6633,-0.0624,-0.1162,-0.0197
2,2,trend_ma200,0.02,0,0.4951,0.6902,-0.1951,0.7563,35,14,21,0.4000,0.0111,-0.0253,0.0067,0.0101,-0.2325,-0.0123,-0.1044,0.1037
3,3,trend_ma200,0.02,20,0.7860,0.6902,0.0959,0.7845,24,9,15,0.3750,0.0211,-0.0288,0.0132,0.0108,-0.1975,0.0055,-0.0791,0.1202
4,4,trend_ma200,0.03,0,0.5039,0.6902,-0.1862,0.7555,33,3,30,0.0909,-0.0000,-0.0379,-0.0000,0.0034,-0.2325,-0.0117,-0.1032,0.1044
5,5,trend_ma200,0.03,20,0.5077,0.6902,-0.1825,0.7618,33,3,30,0.0909,0.0001,-0.0379,0.0001,0.0034,-0.2325,-0.0114,-0.1077,0.1023
6,6,ath_1pct,0.01,0,0.2083,0.6902,-0.4819,0.6195,46,41,4,0.9111,0.0415,-0.0124,0.0037,0.0113,-0.5516,-0.0336,-0.0633,-0.0038
7,7,ath_1pct,0.01,20,0.2978,0.6902,-0.3923,0.8620,25,24,1,0.9600,0.0301,-0.0122,0.0012,0.0117,-0.5682,-0.0264,-0.0430,-0.0108
8,8,ath_1pct,0.02,0,0.2630,0.6902,-0.4271,0.6103,27,21,5,0.8077,0.0323,-0.0219,0.0062,0.0177,-0.5517,-0.0291,-0.0596,-0.0001
9,9,ath_1pct,0.02,20,0.3543,0.6902,-0.3358,0.7614,18,15,3,0.8333,0.0355,-0.0218,0.0059,0.0182,-0.5518,-0.0222,-0.0434,0.0016


In [6]:
"""
Episode scorecard of the slowest candidate of each exit signal (b = 3%, c = 20). S / R > 1 = bought back lower than sold.
"""
# DISPLAY THE TWO SCORECARDS (LARGEST EPISODES FIRST)
scorecard_col_list = ["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "sessions_out", "buy_hold_return_out", "max_decline_out_pct", "open_at_end"]
for exit_signal_str in ["trend_ma200", "ath_1pct"]:
    candidate_scorecard_pdf = exploration_scorecard_pdf[(exploration_scorecard_pdf["exit_signal"] == exit_signal_str) & (exploration_scorecard_pdf["buy_stop"] == 0.03) & (exploration_scorecard_pdf["cooling"] == 20)]
    print(f"\n{exit_signal_str}, b = 3%, c = 20:\t{len(candidate_scorecard_pdf)} episodes, {int((candidate_scorecard_pdf['s_over_r'] > 1).sum())} bought back lower, "
          f"product of S/R {np.prod(candidate_scorecard_pdf['s_over_r']):.4f}")
    display(candidate_scorecard_pdf.sort_values("sessions_out", ascending=False)[scorecard_col_list].head(10).round(4))


trend_ma200, b = 3%, c = 20:	33 episodes, 1 bought back lower, product of S/R 0.8943


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,buy_hold_return_out,max_decline_out_pct,open_at_end
182,2007-12-27,147.39,2009-06-01,94.75,1.5556,358,-0.3573,-0.5448,False
191,2011-08-02,125.53,2011-10-27,128.47,0.9771,61,0.0233,-0.1443,False
190,2010-08-11,109.27,2010-09-13,112.74,0.9692,22,0.0316,-0.0457,False
193,2011-11-09,123.20,2011-12-07,126.65,0.9728,19,0.0278,-0.0559,False
179,2007-11-07,147.64,2007-12-05,148.89,0.9916,19,0.0083,-0.0465,False
170,2006-06-09,125.55,2006-06-29,127.22,0.9869,14,0.0131,-0.0256,False
194,2011-12-08,123.80,2011-12-23,126.37,0.9797,11,0.0206,-0.0305,False
168,2005-10-18,117.90,2005-10-31,120.77,0.9762,9,0.0242,-0.0067,False
171,2006-07-12,125.91,2006-07-25,126.98,0.9916,9,0.0083,-0.0280,False
187,2010-06-04,106.76,2010-06-15,112.02,0.9530,7,0.0491,-0.0199,False



ath_1pct, b = 3%, c = 20:	16 episodes, 5 bought back lower, product of S/R 0.8208


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,buy_hold_return_out,max_decline_out_pct,open_at_end
337,2005-02-11,120.69,2005-07-28,124.48,0.9696,115,0.0312,-0.0588,False
340,2006-02-16,129.03,2006-06-08,126.24,1.0221,77,-0.0218,-0.0468,False
351,2014-07-09,197.08,2014-10-21,193.99,1.0159,73,-0.0158,-0.0770,False
349,2013-12-18,181.68,2014-03-04,187.71,0.9679,50,0.0331,-0.0439,False
342,2006-11-21,140.55,2007-02-02,144.83,0.9704,48,0.0303,-0.0182,False
347,2013-07-11,167.56,2013-09-18,173.05,0.9683,48,0.0326,-0.0270,False
345,2007-10-01,154.46,2007-12-05,148.89,1.0374,46,-0.0362,-0.0886,False
350,2014-04-02,188.91,2014-06-06,195.34,0.9671,45,0.0339,-0.0403,False
344,2007-06-01,153.87,2007-08-06,146.71,1.0488,44,-0.0467,-0.0738,False
346,2013-03-11,156.01,2013-05-03,161.40,0.9666,38,0.0344,-0.0158,False


In [7]:
# SAVE THE RESULTS
write_csv_file_to_path(exploration_pdf, f"{output_path_str}exploration_candidate_data.csv", "W")
write_csv_file_to_path(exploration_scorecard_pdf, f"{output_path_str}exploration_scorecard_data.csv", "W")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp06_capped_regret_reentry/step01_exploration_data/exploration_candidate_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp06_capped_regret_reentry/step01_exploration_data/exploration_candidate_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp06_capped_regret_reentry/step01_exploration_data/exploration_scorecard_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp06_capped_regret_reentry/step01_exploration_data/exploration_scorecard_data.csv' Has Been Saved!


In [8]:
# LOG THE 12 CANDIDATES AS EXPLORATION TRIALS (ONE WRITE)
exploration_trial_pdf = log_trial_pdf(exp_config, "exploration", exploration_pdf.assign(window=f"{exploration_start_str}->{exploration_end_str}"), ["exit_signal", "buy_stop", "cooling", "window"],
                                      [col for col in exploration_pdf.columns if col not in ["candidate_idx", "exit_signal", "exit_order", "buy_stop", "cooling"]], False,
                                      note_str_in="exp06 step01 exploration (2005-2014 continuous replay, context only; data-dependent follow-up of exp03)")
print(f"Trials logged:\t{len(exploration_trial_pdf)}")

Trials logged:	12


In [9]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
1,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
2,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
3,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
4,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
5,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
6,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
7,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
8,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
9,exp03_ath_exit,validation,660,1,0,2026-10-05T14:55:55,2026-10-05T14:56:12
